# The Track

> A Track is the durable record of an Op or Cog execution.
>
> — [@oliphant2026, §5.3]

`track/run-001.trig` records the escalated run as PROV-O [@provo2013]
with EARL outcomes [@earl2017]: three automatic Guard results (passed,
failed, and one honest cantTell), two fired Gates raising two
obligations, and one `earl:manual` approval by a named person that
discharges both obligations and generates the post-review case state
(an action that discharges an obligation must also mutate state, a
ruling logged as GAP-02). SHACL shapes [@shacl2017] derived from the
manifest's own `track.include` list check the record, and refuse the
counterexample in which the obligations were raised and never
discharged. This run's discharger happens to be human, as the two gates
that fired require; the shapes themselves demand a declared mode and a
named accountable party, not humanness, since the paper specifies a
human actor only where the manifest says so (GAP-09 in the
appendix).

One acknowledged judgment call: retention of `final_output` is
conditional on the aggregate outcome (an executed run must retain it, a
noOp run must not). That knowingly departs from the pinned draft's
unconditional include list, and the departure is stated in the shape
messages themselves (GAP-07 in the appendix).

In [1]:
import sys; sys.path[:0] = [".", ".."]  # the repo root, from either cwd
import exhibits
exhibits.check_track_shapes()

run-001               conforms: True
missing-approval      conforms: False
	Message: Obligation raised but not discharged: no discharging action recorded (section 5.6 lists human_approvals in the Track; where the manifest does not name a human the actor is unspecified — provisional: GAP-09).
	Message: Obligation raised but not discharged: no discharging action recorded (section 5.6 lists human_approvals in the Track; where the manifest does not name a human the actor is unspecified — provisional: GAP-09).


> The Track is not just a log file. It is a structured accountability
> artifact.
>
> — [@oliphant2026, §5.3]

Section 5.3 names the purposes a Track serves. On this substrate each
purpose is a query [@sparql2013] over the run graph, not a reading
exercise.

## Auditability

> Auditability — an organization can reconstruct why a decision was
> made.
>
> — [@oliphant2026, §5.3]

In [2]:
exhibits.show_query("auditability.rq");

gate | condition | guardFinding | obliged | evidence | approver | rationale
GATE-01 | confidence < 0.80 | extraction confidence 0.71 is below the 0.80 gate threshold | human review required | Invoice batch 2026-08 (vendor V-2214), 37 invoices | J. Rivera (accounts-payable analyst) | Reviewed the three lowest-confidence extractions against the invoice PDFs; amounts and dates check out. The risk score is driven by the 40 percent month-over-month invoice volume increase; flagging for procurement follow-up is warranted.
GATE-01 | confidence < 0.80 | extraction confidence 0.71 is below the 0.80 gate threshold | human review required | Vendor risk score for V-2214: high | J. Rivera (accounts-payable analyst) | Reviewed the three lowest-confidence extractions against the invoice PDFs; amounts and dates check out. The risk score is driven by the 40 percent month-over-month invoice volume increase; flagging for procurement follow-up is warranted.
GATE-04 | vendor_risk == high | vendor-risk-cog 

## Governance

> Governance — compliance teams can verify that required procedures
> were followed.
>
> — [@oliphant2026, §5.3]

Rows are violations; empty means every obligation raised by a fired
Gate was discharged by a recorded action with a named accountable
party. The same query catches the counterexample:

In [3]:
exhibits.compare_governance()

obligation | obliged | gate
violations on run-001: 0
violations on the counterexample: 2
  https://example.org/vfr/track/run-001#obligation-human-review | human review required | GATE-01
  https://example.org/vfr/track/run-001#obligation-human-approval | human approval required | GATE-04


## Trust

> Trust — users and customers can see that AI work was not merely
> generated, but validated.
>
> — [@oliphant2026, §5.3]

The full outcome distribution, automatic and manual, with cantTell
visible rather than absorbed:

In [4]:
exhibits.show_query("trust.rq");

assertion | label | mode | outcome
https://example.org/vfr/track/run-001#guard-confidence | confidence-guard (in_flight) | http://www.w3.org/ns/earl#automatic | http://www.w3.org/ns/earl#failed
https://example.org/vfr/track/run-001#guard-schema | schema-guard (post_run) | http://www.w3.org/ns/earl#automatic | http://www.w3.org/ns/earl#passed
https://example.org/vfr/track/run-001#guard-source-grounding | source-grounding-guard (post_run) | http://www.w3.org/ns/earl#automatic | http://www.w3.org/ns/earl#cantTell
https://example.org/vfr/track/run-001#guard-vendor-risk-reading | vendor risk reading (in_flight) | http://www.w3.org/ns/earl#automatic | http://www.w3.org/ns/earl#failed
https://example.org/vfr/track/run-001#approval-01 | human review and approval of the vendor flag | http://www.w3.org/ns/earl#manual | http://www.w3.org/ns/earl#passed


## The interface: where every value came from

This purpose is not on the paper's list; it is a consequence of a
ruling logged as GAP-05. The policy applies to oracle-provided values
and is never their provider, so for each variable a gate evaluated the
Track must cite the call: what service, what payload was sent, what
response code came back, and the response that carried the value. The
numerical precision lives inside the oracles or in documented threshold
rules; a reading nobody can source fails the shapes.

In [5]:
exhibits.show_interface()

confidence = 0.71
  service      : extraction-confidence service (invoice-extraction-cog telemetry, v2.3.1)
  sent payload : {"op":"vendor-fraud-review","run":"run-001","variable":"confidence","batch":"invoice-batch-2026-08"}
  responseCode : 200
  response     : {"confidence":0.71}
consensus_disagreement = 0.10
  service      : consensus-comparator service (v1.4.0)
  sent payload : {"op":"vendor-fraud-review","run":"run-001","variable":"consensus_disagreement","cogs":["invoice-extraction-cog","vendor-risk-cog","anomaly-summary-cog"]}
  responseCode : 200
  response     : {"consensus_disagreement":0.10}
sensitive_data_detected = false
  service      : sensitive-data-scanner service (v5.0.2)
  sent payload : {"op":"vendor-fraud-review","run":"run-001","variable":"sensitive_data_detected","batch":"invoice-batch-2026-08"}
  responseCode : 200
  response     : {"sensitive_data_detected":false}
vendor_risk = high
  service      : vendor-risk-cog scoring endpoint (v0.9.7)
  sent payload : {"